[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sheyzafcohen-ui/TaskNest/blob/main/TaskNest.ipynb)

# TaskNest – Automated Support Ticket Router

A local, deterministic support ticket routing pipeline using Python keyword rules and pandas. No LLMs, API keys, or external services are used.

## Routing rules

The classifier lowercases each ticket and scores billing and technical keyword matches using regular expression word boundaries. The higher score wins, a non-zero tie goes to Technical, and no matches go to General. General also covers feature requests, general questions, and thank-you messages.

In [ ]:
import re
import pandas as pd

# Only the Python standard library and pandas are used.
BILLING_KEYWORDS = (
    "payment", "payments", "invoice", "invoices", "refund",
    "refunds", "subscription", "subscriptions", "billing",
    "charge", "charged", "receipt", "cancel plan",
)
TECHNICAL_KEYWORDS = (
    "technical", "issue", "issues", "crash", "crashes",
    "crashing", "crashed", "error", "error message", "bug", "login",
    "log in", "sign in", "password", "freeze", "freezing",
    "broken", "not working",
)

CATEGORY_DETAILS = {
    "billing": {
        "team": "Billing",
        "reply": "Thank you for reaching out. Your inquiry has been routed to our Billing team, and a specialist will review your account shortly.",
    },
    "technical": {
        "team": "Tech",
        "reply": "Thank you for contacting us. We have forwarded your issue to our Tech team, and an engineer will assist you soon.",
    },
    "general": {
        "team": "General",
        "reply": "Thank you for your message. Your inquiry has been assigned to our General team, and we will get back to you shortly.",
    },
}

def _keyword_score(text: str, keywords: tuple[str, ...]) -> int:
    """Count distinct configured keywords present as whole words or phrases."""
    return sum(
        bool(re.search(r"\b" + re.escape(keyword) + r"\b", text))
        for keyword in keywords
    )

def classify_ticket(ticket: str) -> str:
    """Classify a ticket using keyword scores and a technical tie-break."""
    text = str(ticket).lower()
    billing_score = _keyword_score(text, BILLING_KEYWORDS)
    technical_score = _keyword_score(text, TECHNICAL_KEYWORDS)

    if billing_score == 0 and technical_score == 0:
        return "general"
    if technical_score >= billing_score:
        return "technical"
    return "billing"

## Process tickets

Each result keeps the original ticket, category, assigned team, and fixed customer reply in the requested column order.

In [ ]:
def process_tickets(tickets: list[str]) -> pd.DataFrame:
    """Route each ticket and return the four-column results DataFrame."""
    rows = []
    for ticket in tickets:
        category = classify_ticket(ticket)
        details = CATEGORY_DETAILS[category]
        rows.append({
            "ticket": ticket,
            "category": category,
            "team": details["team"],
            "reply": details["reply"],
        })
    return pd.DataFrame(rows, columns=["ticket", "category", "team", "reply"])

## Sample tickets, verification, and export

The examples include three tickets for each ordinary category and one billing/technical tie. The assertions verify the tie-break, category totals, and exact output schema. The final cell writes `tasknest_routed_tickets.csv` to the current directory.

In [ ]:
sample_tickets = [
    "I need help understanding the payment on my latest statement.",
    "Could you email me the invoices for the last two months?",
    "Please issue refunds for the duplicate subscription charge.",
    "I have a technical issue opening my account settings.",
    "The app crashes every time I try to save a document.",
    "I see an error message when I sign in with my password.",
    "Please consider adding a dark mode feature request.",
    "What are your support hours on weekends?",
    "Thank you for helping me resolve my question so quickly!",
    "My payment failed because the app crashed",
]

routed_tickets = process_tickets(sample_tickets)
assert list(routed_tickets.columns) == ["ticket", "category", "team", "reply"]
assert routed_tickets["category"].value_counts().to_dict() == {"billing": 3, "technical": 4, "general": 3}
assert classify_ticket("My payment failed because the app crashed") == "technical"

display(routed_tickets)
print("\nTicket count by team:")
print(routed_tickets["team"].value_counts().reindex(["Billing", "Tech", "General"], fill_value=0))

routed_tickets.to_csv("tasknest_routed_tickets.csv", index=False)
print("\nExported tasknest_routed_tickets.csv")